# SkillPulse — Setup & Data Collection
Run cells top to bottom. Make sure `.env` and MySQL are set up first (see README notes).

## 1. Imports

In [16]:
import json as pyjson

In [15]:
import os
import time
import requests
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from urllib.parse import quote_plus

load_dotenv(dotenv_path="../.env")  # reads variables from .env in the same folder


True

## 2. Connect to MySQL
This reads credentials from `.env` — never hardcode them here.

In [17]:

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

#engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
engine = create_engine(f"mysql+pymysql://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
# quick connection test
with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES;"))
    print("Connected. Tables found:", [row[0] for row in result])


Connected. Tables found: ['drift_reports', 'job_postings', 'job_skills', 'model_runs', 'skills']


## 3. Adzuna scraper function
Pulls one page of results at a time.

In [18]:
ADZUNA_APP_ID = os.getenv("ADZUNA_APP_ID")
ADZUNA_APP_KEY = os.getenv("ADZUNA_APP_KEY")

def fetch_page(page, country="in", keyword="data scientist", results_per_page=50):
    url = f"https://api.adzuna.com/v1/api/jobs/{country}/search/{page}"
    params = {
        "app_id": ADZUNA_APP_ID,
        "app_key": ADZUNA_APP_KEY,
        "results_per_page": results_per_page,
        "what": keyword,
        "content-type": "application/json",
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json().get("results", [])


## 4. Test pull — small batch first
Check salary-field coverage before running the full 5,000–10,000 pull.

In [19]:
test_results = fetch_page(page=1, keyword="data scientist", results_per_page=50)
df_test = pd.json_normalize(test_results)
print("Rows pulled:", len(df_test))

for col in ["salary_min", "salary_max"]:
    if col not in df_test.columns:
        df_test[col] = pd.NA

print("Salary fields populated:", df_test[["salary_min", "salary_max"]].notna().mean())
df_test.head()

Rows pulled: 50
Salary fields populated: salary_min    0.0
salary_max    0.0
dtype: float64


,description,title,redirect_url,created,salary_is_predicted,id,adref,longitude,__CLASS__,contract_time,...,company.display_name,category.__CLASS__,category.label,category.tag,location.display_name,location.__CLASS__,location.area,contract_type,salary_min,salary_max
0,Data Science Instructor: NDMIT is seeking a pa...,Data Science Instructor,https://www.adzuna.in/land/ad/5795582923?se=ng...,2026-07-10T17:48:43Z,0,5795582923,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTc5NTU4MjkyMyIsI...,78.06485,Adzuna::API::Response::Job,full_time,...,NDMIT,Adzuna::API::Response::Category,IT Jobs,it-jobs,"Agra, Uttar Pradesh",Adzuna::API::Response::Location,"[India, Uttar Pradesh, Agra]",NaN,<NA>,<NA>
1,Data Science & AI Instructor upGrad is expandi...,Data Science Instructor,https://www.adzuna.in/land/ad/5786785503?se=ng...,2026-07-03T17:37:09Z,0,5786785503,eyJhbGciOiJIUzI1NiJ9.eyJzIjoibmc3VGlRcUE4Ukdue...,NaN,Adzuna::API::Response::Job,NaN,...,upGrad,Adzuna::API::Response::Category,IT Jobs,it-jobs,India,Adzuna::API::Response::Location,[India],NaN,<NA>,<NA>
2,"This job is with Warner Bros. Discovery, an in...",Principal Data Scientist,https://www.adzuna.in/details/5785771652?utm_m...,2026-07-03T00:54:19Z,0,5785771652,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTc4NTc3MTY1MiIsI...,78.50806,Adzuna::API::Response::Job,full_time,...,Warner Bros. Discovery,Adzuna::API::Response::Category,IT Jobs,it-jobs,"Hyderabad, Telangana",Adzuna::API::Response::Location,"[India, Telangana, Hyderabad]",NaN,<NA>,<NA>
3,"This job is with S&P Global, an inclusive empl...",Data Scientist III,https://www.adzuna.in/details/5785770925?utm_m...,2026-07-03T00:54:12Z,0,5785770925,eyJhbGciOiJIUzI1NiJ9.eyJzIjoibmc3VGlRcUE4Ukdue...,78.50806,Adzuna::API::Response::Job,full_time,...,S&P Global,Adzuna::API::Response::Category,IT Jobs,it-jobs,"Hyderabad, Telangana",Adzuna::API::Response::Location,"[India, Telangana, Hyderabad]",NaN,<NA>,<NA>
4,Job Title: Data Science Instructor – Offline L...,Data Science Instructor,https://www.adzuna.in/land/ad/5785410815?se=ng...,2026-07-02T17:24:24Z,0,5785410815,eyJhbGciOiJIUzI1NiJ9.eyJzIjoibmc3VGlRcUE4Ukdue...,NaN,Adzuna::API::Response::Job,NaN,...,upGrad,Adzuna::API::Response::Category,IT Jobs,it-jobs,India,Adzuna::API::Response::Location,[India],NaN,<NA>,<NA>


In [20]:
import json
print(json.dumps(test_results[0], indent=2))

{
  "description": "Data Science Instructor: NDMIT is seeking a passionate and knowledgeable Data Science Instructor to join our Kanpur branch. The ideal candidate should have hands-on experience in Data Science concepts and the ability to train students through practical and industry-oriented sessions. Key Responsibilities: \u2022 Deliver engaging classroom and practical sessions on Data Science. \u2022 Train students in Python, Statistics, Machine Learning, Deep Learning, SQL, and Power BI. \u2022 Guide students on real-time p\u2026",
  "title": "Data Science Instructor",
  "redirect_url": "https://www.adzuna.in/land/ad/5795582923?se=ng7TiQqA8RGnybZGKZMVYw&utm_medium=api&utm_source=7a91684d&v=B16697C360F096A495E962C02DC9FC0339F3B405",
  "created": "2026-07-10T17:48:43Z",
  "company": {
    "__CLASS__": "Adzuna::API::Response::Company",
    "display_name": "NDMIT"
  },
  "category": {
    "__CLASS__": "Adzuna::API::Response::Category",
    "label": "IT Jobs",
    "tag": "it-jobs"
  },

In [21]:
salary_count = sum(1 for r in test_results if r.get("salary_min") is not None)
print(f"Postings with salary_min present: {salary_count} out of {len(test_results)}")

Postings with salary_min present: 0 out of 50


In [22]:
# Test salary coverage across different countries
for country_code in ["gb", "us"]:
    results = fetch_page(page=1, keyword="data scientist", results_per_page=50, country=country_code)
    salary_count = sum(1 for r in results if r.get("salary_min") is not None)
    print(f"{country_code}: {salary_count} out of {len(results)} postings have salary_min")

gb: 50 out of 50 postings have salary_min
us: 50 out of 50 postings have salary_min


## 5. Insert into MySQL
Writes each result into the `job_postings` table.

In [23]:
def insert_postings(results, source="adzuna", country="gb"):
    rows = []
    for r in results:
        rows.append({
            "source": source,
            "country": country,
            "title": r.get("title"),
            "company": r.get("company", {}).get("display_name"),
            "location": r.get("location", {}).get("display_name"),
            "salary_min": r.get("salary_min"),
            "salary_max": r.get("salary_max"),
            "description": r.get("description"),
            "posted_date": r.get("created", "")[:10] or None,
            "raw_json": pyjson.dumps(r),
        })
    df = pd.DataFrame(rows)
    df.to_sql("job_postings", con=engine, if_exists="append", index=False)
    print(f"Inserted {len(df)} rows ({country})")

## 6. Full paginated pull
Loops through multiple pages with rate-limit delay. Adjust `max_pages` and `keywords` as needed.

In [24]:
countries = ["in", "gb", "us"]
keywords = ["data scientist", "software engineer", "machine learning engineer", "backend developer"]
max_pages = 10

for country_code in countries:
    for kw in keywords:
        for page in range(1, max_pages + 1):
            try:
                results = fetch_page(page=page, keyword=kw, results_per_page=50, country=country_code)
                if not results:
                    break
                insert_postings(results, source="adzuna", country=country_code)
                time.sleep(1)
            except Exception as e:
                print(f"Error on {country_code}/{kw} page {page}: {e}")
                time.sleep(5)

Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (gb)
Inserted 50 rows (gb)
Inserted 50 rows (gb)
Inserted 50 rows (gb)
Inserted 50 rows (gb)
Inserted 5

In [27]:
with engine.connect() as conn:
    result = conn.execute(text("DELETE FROM job_postings WHERE country IS NULL"))
    conn.commit()
    print(f"Deleted {result.rowcount} rows with missing country")

Deleted 4050 rows with missing country


## 7. Sanity check
Confirm what landed in the database.

In [28]:
df_check = pd.read_sql(
    "SELECT country, COUNT(*) AS total, SUM(salary_min IS NOT NULL) AS with_salary FROM job_postings GROUP BY country;",
    con=engine
)
df_check

,country,total,with_salary
0,in,2000,834.0
1,gb,2000,1999.0
2,us,2000,2000.0
